# Chapter 6 - Semantic segmentation with a U-Net

Companion to [`docs/06_segmentation.md`](../docs/06_segmentation.md).

> **GPU: Runtime -> Change runtime type -> T4 GPU.** About 6 minutes total (one 20-epoch run
> plus three short ablations, then a ~160 MB pretrained-model download at the end).

We use a **synthetic shapes** dataset generated in-process: no download, instant, and - crucially -
we know the ground truth exactly, so when something looks wrong we know it *is* wrong. Real
datasets come at the end, via a pretrained DeepLab.

The plan:

1. Build the data, and look at the **class imbalance** that makes pixel accuracy useless.
2. Implement IoU / Dice, and prove that a do-nothing model scores 94% pixel accuracy.
3. Meet the two mask pitfalls (interpolation, paired augmentation) before they bite.
4. Build a U-Net from scratch. Train it with CE + Dice.
5. **Ablate the skip connections** to see exactly what they buy.
6. Run a pretrained DeepLabV3 on a real photo.

In [ ]:
import sys, time, math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw

print('torch', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    print('\n*** NO GPU: Runtime -> Change runtime type -> T4 GPU, then Restart. ***')
print('device', device)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
NUM_WORKERS = 2 if IN_COLAB else 0

def set_seed(seed=0):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(0)
torch.backends.cudnn.benchmark = True
plt.rcParams['figure.dpi'] = 110

CLASS_NAMES = ['background', 'circle', 'rectangle', 'triangle']
N_CLASSES = len(CLASS_NAMES)
MASK_CMAP = ListedColormap(['#1a1a2e', '#e94560', '#4ecca3', '#4d96ff'])
print('classes:', dict(enumerate(CLASS_NAMES)))

## 1. The dataset

Each sample: a noisy RGB image and an integer mask of the same size, where pixel value `k` means
"this pixel belongs to class `k`". Shapes overlap, so the model has to handle occlusion.

Note the two shapes we return: image `(3, H, W)` float32, mask `(H, W)` **int64**.

In [ ]:
IMG_SIZE = 128

def make_sample(rng, size=IMG_SIZE, n_shapes=(2, 5), noise=0.06):
    """-> (image HWC float32 0..1, mask HW int64 with values 0..3)"""
    bg = int(rng.integers(20, 70))
    img = Image.new('RGB', (size, size), (bg, bg, int(bg * 1.2)))
    mask = Image.new('L', (size, size), 0)
    d, dm = ImageDraw.Draw(img), ImageDraw.Draw(mask)

    for _ in range(int(rng.integers(*n_shapes))):
        kind = int(rng.integers(1, 4))                       # 1=circle 2=rect 3=triangle
        w = int(rng.integers(size // 6, size // 2))
        h = int(rng.integers(size // 6, size // 2))
        x0 = int(rng.integers(0, size - w))
        y0 = int(rng.integers(0, size - h))
        box = [x0, y0, x0 + w, y0 + h]
        colour = tuple(int(c) for c in rng.integers(110, 255, size=3))
        if kind == 1:
            d.ellipse(box, fill=colour);   dm.ellipse(box, fill=kind)
        elif kind == 2:
            d.rectangle(box, fill=colour); dm.rectangle(box, fill=kind)
        else:
            tri = [(x0, y0 + h), (x0 + w // 2, y0), (x0 + w, y0 + h)]
            d.polygon(tri, fill=colour);   dm.polygon(tri, fill=kind)

    arr = np.asarray(img, dtype=np.float32) / 255.0
    arr = np.clip(arr + rng.normal(0, noise, arr.shape).astype(np.float32), 0, 1)
    return arr, np.asarray(mask, dtype=np.int64)


class ShapesDataset(Dataset):
    """Pre-generates everything into memory: tiny, and removes the loader from the equation."""

    def __init__(self, n, seed=0, augment=False):
        rng = np.random.default_rng(seed)
        self.items = [make_sample(rng) for _ in range(n)]
        self.augment = augment
        self.rng = np.random.default_rng(seed + 9999)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        img, mask = self.items[i]
        if self.augment:
            img, mask = paired_augment(img, mask, self.rng)
        x = torch.from_numpy(np.ascontiguousarray(img.transpose(2, 0, 1)))   # HWC -> CHW
        y = torch.from_numpy(np.ascontiguousarray(mask))                     # int64, NOT one-hot
        return x, y


def paired_augment(img, mask, rng):
    """Geometric augmentation applied IDENTICALLY to image and mask."""
    if rng.random() < 0.5:
        img, mask = img[:, ::-1], mask[:, ::-1]          # horizontal flip
    if rng.random() < 0.5:
        img, mask = img[::-1], mask[::-1]                # vertical flip
    k = int(rng.integers(0, 4))
    if k:
        img, mask = np.rot90(img, k, axes=(0, 1)), np.rot90(mask, k, axes=(0, 1))
    return np.ascontiguousarray(img), np.ascontiguousarray(mask)


t0 = time.perf_counter()
train_ds = ShapesDataset(600, seed=0, augment=True)
val_ds = ShapesDataset(150, seed=1234, augment=False)
print(f'generated {len(train_ds)} train + {len(val_ds)} val samples in {time.perf_counter() - t0:.1f}s')

x, y = train_ds[0]
print(f'\nimage {tuple(x.shape)} {x.dtype} range ({x.min():.2f}, {x.max():.2f})')
print(f'mask  {tuple(y.shape)} {y.dtype}  values present: {sorted(set(y.numpy().ravel().tolist()))}')
print('\nThe mask is an INTEGER map, not one-hot. nn.CrossEntropyLoss wants exactly this.')

BATCH = 16
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=device.type == 'cuda', drop_last=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=device.type == 'cuda')
xb, yb = next(iter(train_loader))
print(f'\nbatch: x {tuple(xb.shape)} | y {tuple(yb.shape)} {yb.dtype}')
print('target has NO channel axis - that is the shape CrossEntropyLoss expects for segmentation.')

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(13, 6.6))
for col in range(6):
    img, mask = val_ds.items[col]
    axes[0, col].imshow(img)
    axes[1, col].imshow(mask, cmap=MASK_CMAP, vmin=0, vmax=3)
    overlay = img.copy()
    for cls in range(1, N_CLASSES):
        sel = mask == cls
        colour = np.array(MASK_CMAP(cls)[:3], dtype=np.float32)
        overlay[sel] = 0.55 * overlay[sel] + 0.45 * colour
    axes[2, col].imshow(overlay)
for ax, lab in zip(axes[:, 0], ['image', 'mask', 'overlay']):
    ax.set_ylabel(lab, fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('synthetic shapes: image, ground-truth mask, overlay')
plt.tight_layout()

all_masks = np.stack([m for _, m in val_ds.items])
counts = np.bincount(all_masks.ravel(), minlength=N_CLASSES)
print('pixel class balance over the validation set:')
for i, n in enumerate(counts):
    print(f'  {i} {CLASS_NAMES[i]:11} {n:9,d} px  {100 * n / counts.sum():5.2f}%')
print(f'\nBackground is {100 * counts[0] / counts.sum():.1f}% of all pixels.')
print('Remember that number when you see the pixel-accuracy demo below.')

## 2. Metrics: why pixel accuracy is a trap

Build the confusion matrix once (with chapter 1's `bincount` trick, now over millions of pixels),
then read every metric off it.

$$\text{IoU} = \frac{TP}{TP+FP+FN}, \qquad \text{Dice} = \frac{2TP}{2TP+FP+FN}$$

**Neither counts true negatives** - which is precisely why they can't be fooled by predicting
background.

In [ ]:
def confusion_matrix(target, pred, k=N_CLASSES):
    """Rows actual, cols predicted. One bincount pass over all pixels."""
    t = np.asarray(target).ravel().astype(np.int64)
    p = np.asarray(pred).ravel().astype(np.int64)
    return np.bincount(t * k + p, minlength=k * k).reshape(k, k)

def metrics_from_cm(cm):
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    with np.errstate(divide='ignore', invalid='ignore'):
        iou = np.where(tp + fp + fn > 0, tp / (tp + fp + fn), np.nan)
        dice = np.where(2 * tp + fp + fn > 0, 2 * tp / (2 * tp + fp + fn), np.nan)
    return {'pixel_acc': float(tp.sum() / cm.sum()),
            'iou': iou, 'miou': float(np.nanmean(iou)),
            'dice': dice, 'mdice': float(np.nanmean(dice))}


lazy_pred = np.zeros_like(all_masks)                       # "everything is background"
m_lazy = metrics_from_cm(confusion_matrix(all_masks, lazy_pred))

print('A model that predicts BACKGROUND for every single pixel:')
print(f'  pixel accuracy {m_lazy["pixel_acc"]:.4f}   <- looks like a working model!')
print(f'  mIoU           {m_lazy["miou"]:.4f}   <- the truth')
print('  per-class IoU  ', np.round(m_lazy['iou'], 4))
print('\nPerfect prediction, for reference:')
m_perfect = metrics_from_cm(confusion_matrix(all_masks, all_masks))
print(f'  pixel accuracy {m_perfect["pixel_acc"]:.4f} | mIoU {m_perfect["miou"]:.4f}')

print('\nand a coarse-but-roughly-right prediction (the truth downsampled 8x and back,')
print('which is roughly what a decoder with no skip connections produces):')
_t = torch.from_numpy(all_masks)[:, None].float()
coarse = F.interpolate(F.interpolate(_t, scale_factor=0.125, mode='nearest'),
                       scale_factor=8, mode='nearest')[:, 0].numpy().astype(np.int64)
m_coarse = metrics_from_cm(confusion_matrix(all_masks, coarse))
print(f'  pixel accuracy {m_coarse["pixel_acc"]:.4f} | mIoU {m_coarse["miou"]:.4f}')
print('\nPixel accuracy hardly distinguishes "does nothing" from "roughly right":')
print(f'  {m_lazy["pixel_acc"]:.3f} -> {m_coarse["pixel_acc"]:.3f} -> {m_perfect["pixel_acc"]:.3f}')
print(f'mIoU separates them properly:')
print(f'  {m_lazy["miou"]:.3f} -> {m_coarse["miou"]:.3f} -> {m_perfect["miou"]:.3f}')
print('Report mIoU and per-class IoU. Always.')

print('\nIoU and Dice are monotonically related: dice = 2*iou/(1+iou)')
for iou_v in [0.2, 0.5, 0.8]:
    print(f'  IoU {iou_v:.2f} -> Dice {2 * iou_v / (1 + iou_v):.3f}')
print('So they rank models identically. Dice is just the more flattering number.')

## 3. Two pitfalls, demonstrated before they bite you

### Pitfall 1: never interpolate a mask bilinearly

In [ ]:
small_mask = val_ds.items[0][1]
t = torch.from_numpy(small_mask)[None, None].float()

bilinear = F.interpolate(t, scale_factor=0.5, mode='bilinear', align_corners=False)[0, 0].numpy()
nearest = F.interpolate(t, scale_factor=0.5, mode='nearest')[0, 0].numpy()

print('original mask values  :', sorted(set(small_mask.ravel().tolist())))
print('after BILINEAR /2     :', np.round(sorted(set(bilinear.ravel().tolist()))[:9], 3), '...')
print('after NEAREST  /2     :', sorted(set(nearest.ravel().tolist())))
print(f'\nbilinear invented {len(set(bilinear.ravel().tolist()))} distinct values from {len(set(small_mask.ravel().tolist()))}.')
print('A value of 2.4 becomes class 2 when you cast it - a class that was never there.')
print('Between a circle (1) and a triangle (3) you now get rectangles (2). Out of nothing.')

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(small_mask, cmap=MASK_CMAP, vmin=0, vmax=3); axes[0].set_title('original mask')
axes[1].imshow(bilinear, cmap=MASK_CMAP, vmin=0, vmax=3); axes[1].set_title('BILINEAR (wrong)')
axes[2].imshow(nearest, cmap=MASK_CMAP, vmin=0, vmax=3); axes[2].set_title('NEAREST (correct)')
axes[3].imshow(np.abs(bilinear - nearest) > 0.01, cmap='Reds'); axes[3].set_title('where they differ')
for ax in axes: ax.axis('off')
plt.tight_layout()
print('\nRule: images resize bilinear, masks resize NEAREST. No exceptions.')

### Pitfall 2: augmentation must transform image and mask together

In [ ]:
img0, mask0 = val_ds.items[1]
rng_a = np.random.default_rng(7)
rng_b = np.random.default_rng(99)

good_img, good_mask = paired_augment(img0, mask0, np.random.default_rng(5))
bad_img, _ = paired_augment(img0, mask0, rng_a)             # image with one random draw
_, bad_mask = paired_augment(img0, mask0, rng_b)            # mask with a DIFFERENT draw

def agreement(img, mask):
    """Fraction of shape pixels where the image is brighter than the background - a proxy for
    'does the mask line up with the picture'."""
    bright = img.mean(-1) > 0.35
    fg = mask > 0
    return float((bright & fg).sum() / max(fg.sum(), 1))

fig, axes = plt.subplots(2, 3, figsize=(9.5, 6))
axes[0, 0].imshow(good_img); axes[0, 0].set_title('image (paired)', fontsize=9)
axes[0, 1].imshow(good_mask, cmap=MASK_CMAP, vmin=0, vmax=3); axes[0, 1].set_title('mask (paired)', fontsize=9)
axes[0, 2].imshow(good_img); axes[0, 2].imshow(good_mask > 0, alpha=0.4, cmap='Reds')
axes[0, 2].set_title(f'aligned: {agreement(good_img, good_mask):.2f}', fontsize=9)
axes[1, 0].imshow(bad_img); axes[1, 0].set_title('image (own transform)', fontsize=9)
axes[1, 1].imshow(bad_mask, cmap=MASK_CMAP, vmin=0, vmax=3); axes[1, 1].set_title('mask (different transform)', fontsize=9)
axes[1, 2].imshow(bad_img); axes[1, 2].imshow(bad_mask > 0, alpha=0.4, cmap='Reds')
axes[1, 2].set_title(f'MISALIGNED: {agreement(bad_img, bad_mask):.2f}', fontsize=9)
for ax in axes.ravel(): ax.axis('off')
plt.tight_layout()

print(f'agreement, paired transforms     : {agreement(good_img, good_mask):.3f}')
print(f'agreement, independent transforms: {agreement(bad_img, bad_mask):.3f}')
print('\nThe bottom row is what you get from two separate transforms.Compose calls with')
print('random parameters. It runs. The loss even decreases (the model learns the average')
print('mask). Nothing errors. Your mIoU is just bad forever.')
print('\nThree ways to do it right:')
print('  1. torchvision.transforms.v2 - designed to transform image+mask together')
print('  2. albumentations - the standard library for this')
print('  3. write the geometry by hand, as paired_augment above does')

## 4. The U-Net

Encoder downsamples and grows channels; decoder upsamples and shrinks; **skip connections**
concatenate encoder features into the matching decoder stage.

Three pooling levels means the input must be divisible by 8. We keep a size guard in `Up` anyway,
because real inputs are never the size you want.

In [ ]:
class DoubleConv(nn.Module):
    """(conv 3x3 -> BN -> ReLU) x2 - the U-Net building block."""

    def __init__(self, c_in, c_out):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Up(nn.Module):
    """Upsample, concatenate the skip, then DoubleConv."""

    def __init__(self, c_in, c_skip, c_out, mode='bilinear'):
        super().__init__()
        if mode == 'bilinear':
            self.up = nn.Sequential(
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                nn.Conv2d(c_in, c_in // 2, 3, padding=1, bias=False))
        else:
            self.up = nn.ConvTranspose2d(c_in, c_in // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(c_in // 2 + c_skip, c_out)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:                 # odd input sizes land here
            x = F.interpolate(x, size=skip.shape[-2:], mode='nearest')
        return self.conv(torch.cat([x, skip], dim=1))        # concatenate along CHANNELS


class UNet(nn.Module):
    def __init__(self, n_classes=N_CLASSES, c_in=3, base=16, mode='bilinear', use_skips=True):
        super().__init__()
        b = base
        self.use_skips = use_skips
        self.enc1 = DoubleConv(c_in, b)          # 128 -> 128,  b ch
        self.enc2 = DoubleConv(b, 2 * b)         #  64 ->  64, 2b ch
        self.enc3 = DoubleConv(2 * b, 4 * b)     #  32 ->  32, 4b ch
        self.bottleneck = DoubleConv(4 * b, 8 * b)   # 16 -> 16, 8b ch
        self.pool = nn.MaxPool2d(2)
        self.up3 = Up(8 * b, 4 * b, 4 * b, mode)
        self.up2 = Up(4 * b, 2 * b, 2 * b, mode)
        self.up1 = Up(2 * b, b, b, mode)
        self.out = nn.Conv2d(b, n_classes, kernel_size=1)    # 1x1: per-pixel classifier

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        bo = self.bottleneck(self.pool(e3))
        if not self.use_skips:                    # ablation: zeros keep the shapes and the
            e1, e2, e3 = (torch.zeros_like(t) for t in (e1, e2, e3))   # parameter count identical
        d3 = self.up3(bo, e3)
        d2 = self.up2(d3, e2)
        d1 = self.up1(d2, e1)
        return self.out(d1)                       # (N, n_classes, H, W) LOGITS


model = UNet().to(device)
print(f'parameters: {sum(p.numel() for p in model.parameters()):,}\n')

with torch.no_grad():
    x = torch.randn(2, 3, 128, 128, device=device)
    e1 = model.enc1(x); e2 = model.enc2(model.pool(e1)); e3 = model.enc3(model.pool(e2))
    bo = model.bottleneck(model.pool(e3))
    print('encoder path (resolution down, channels up):')
    for nm, t in [('input', x), ('enc1', e1), ('enc2', e2), ('enc3', e3), ('bottleneck', bo)]:
        print(f'  {nm:11} {tuple(t.shape)}')
    d3 = model.up3(bo, e3); d2 = model.up2(d3, e2); d1 = model.up1(d2, e1)
    print('decoder path (resolution up, channels down):')
    for nm, t in [('up3', d3), ('up2', d2), ('up1', d1), ('out', model.out(d1))]:
        print(f'  {nm:11} {tuple(t.shape)}')

print('\nThe output is (N, 4, 128, 128): one logit per class PER PIXEL.')
print('Compare with chapter 4, where the head collapsed everything to (N, 10).')

In [ ]:
print('output size must equal input size:')
for s in [128, 64, 256]:
    with torch.no_grad():
        o = model(torch.randn(1, 3, s, s, device=device))
    print(f'  input {s:3d}x{s:3d} -> output {tuple(o.shape[-2:])}  match: {o.shape[-1] == s}')

print('\nawkward sizes (not divisible by 8) - this is where segmentation code breaks:')
for s in [100, 130]:
    with torch.no_grad():
        o = model(torch.randn(1, 3, s, s, device=device))
    print(f'  input {s}x{s} -> output {tuple(o.shape[-2:])}  match: {o.shape[-1] == s}')
print('\nOurs survives because Up interpolates to the skip size. Without that guard you get a')
print('shape error on torch.cat, or - worse - an output a few pixels smaller than the target,')
print('and CrossEntropyLoss will refuse it. The clean fix on real data is to pad the input to a')
print('multiple of your total downsampling factor, predict, then crop back.')

## 5. Losses

Cross-entropy handles `(N, C, H, W)` vs `(N, H, W)` natively. Dice loss optimizes the overlap
metric directly, using **softmax probabilities** - `argmax` has no gradient.

In [ ]:
ce = nn.CrossEntropyLoss()

def dice_loss(logits, target, eps=1.0):
    """Soft multi-class Dice loss. logits (N,C,H,W), target (N,H,W) int64."""
    n_classes = logits.shape[1]
    # .float() matters under AMP: we sum over N*H*W (~260k) elements, and float16 saturates at
    # 65504, so an fp16 sum would overflow to inf and the loss would become nan.
    probs = torch.softmax(logits.float(), dim=1)                          # differentiable
    target_1h = F.one_hot(target, n_classes).permute(0, 3, 1, 2).float()  # (N,H,W) -> (N,C,H,W)
    dims = (0, 2, 3)                                                      # sum over batch+space
    inter = (probs * target_1h).sum(dims)
    denom = probs.sum(dims) + target_1h.sum(dims)
    dice = (2 * inter + eps) / (denom + eps)                              # (C,)
    return 1 - dice.mean()

def combined_loss(logits, target):
    return ce(logits, target) + dice_loss(logits, target)


xb_d, yb_d = xb.to(device), yb.to(device)
with torch.no_grad():
    logits = model(xb_d)
print('shapes into the loss:')
print(f'  logits {tuple(logits.shape)} {logits.dtype}   <- (N, C, H, W) float32, RAW')
print(f'  target {tuple(yb_d.shape)} {yb_d.dtype}    <- (N, H, W) int64, class indices')
print(f'\nCE loss       {ce(logits, yb_d).item():.4f}   (untrained; ln(4) = {math.log(4):.4f})')
print(f'Dice loss     {dice_loss(logits, yb_d).item():.4f}')
print(f'combined      {combined_loss(logits, yb_d).item():.4f}')

print('\nthe common mistakes, made explicit:')
try:
    ce(logits, yb_d.float())
    print('  float class indices -> no error (unexpected on this torch version)')
except (RuntimeError, TypeError) as e:
    print('  float class indices ->', str(e).split('\n')[0][:92])
try:
    ce(logits, yb_d[:, None])
    print('  target with a channel axis -> no error (unexpected)')
except (RuntimeError, TypeError) as e:
    print('  target (N,1,H,W)    ->', str(e).split('\n')[0][:92])

one_hot_loss = ce(logits, F.one_hot(yb_d, N_CLASSES).permute(0, 3, 1, 2).float())
print(f'\n  one-hot float target -> {one_hot_loss.item():.4f}, and NO error. Worth understanding:')
print('  since PyTorch 1.10 a float target of the same shape as the input is treated as class')
print(f'  PROBABILITIES (soft labels). For an exact one-hot it equals the index form '
      f'({ce(logits, yb_d).item():.4f}).')
print('  So this one is not a bug - but it means something different, and it is how you do')
print('  label smoothing or knowledge distillation on a segmentation task.')
print('\n  softmax then CE     -> no error at all, just a worse model. Same trap as chapter 2.')
print('\nignore_index for unlabelled pixels (255 is the usual convention):')
masked = yb_d.clone(); masked[:, :10, :] = 255
print(f'  CrossEntropyLoss(ignore_index=255) = {nn.CrossEntropyLoss(ignore_index=255)(logits, masked).item():.4f}')
print('  Essential on real datasets, where boundary pixels are marked void rather than guessed.')

## 6. Train

In [ ]:
def make_scaler(enabled):
    try:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def train_epoch(model, loader, optimizer, loss_fn, scaler, scheduler=None):
    model.train()
    tot, seen = 0.0, 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=scaler.is_enabled()):
            loss = loss_fn(model(xb), yb)
        if scaler.is_enabled():
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); optimizer.step()
        tot += loss.item() * xb.size(0)
        seen += xb.size(0)
    if scheduler is not None:
        scheduler.step()
    return tot / seen


@torch.no_grad()
def evaluate_seg(model, loader, loss_fn):
    """Accumulate ONE confusion matrix over the whole set, then derive every metric."""
    model.eval()
    cm = np.zeros((N_CLASSES, N_CLASSES), dtype=np.int64)
    tot, seen = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        logits = model(xb)
        tot += loss_fn(logits, yb).item() * xb.size(0)
        seen += xb.size(0)
        pred = logits.argmax(1)                            # (N,C,H,W) -> (N,H,W)
        cm += confusion_matrix(yb.cpu().numpy(), pred.cpu().numpy())
    m = metrics_from_cm(cm)
    m['loss'] = tot / seen
    m['cm'] = cm
    return m

print('defined train_epoch and evaluate_seg')
print('\nNote evaluate_seg accumulates a confusion matrix across batches rather than averaging')
print('per-batch IoU. Per-batch averaging is WRONG: a batch missing a class contributes a')
print('0/0 for it, and the average of ratios is not the ratio of sums.')

In [ ]:
EPOCHS = 20
set_seed(0)
model = UNet(base=16).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = make_scaler(device.type == 'cuda')

print(f'training {EPOCHS} epochs, CE + Dice, AdamW 3e-3 cosine')
print(f'{"ep":>3} {"train_loss":>11} {"val_loss":>9} {"pix_acc":>8} {"mIoU":>7}   per-class IoU')
history = {'train_loss': [], 'val_loss': [], 'miou': [], 'pixel_acc': [], 'iou': []}
best_miou, best_state = 0.0, None
t_start = time.perf_counter()

for ep in range(EPOCHS):
    tr = train_epoch(model, train_loader, optimizer, combined_loss, scaler, scheduler)
    m = evaluate_seg(model, val_loader, combined_loss)
    history['train_loss'].append(tr); history['val_loss'].append(m['loss'])
    history['miou'].append(m['miou']); history['pixel_acc'].append(m['pixel_acc'])
    history['iou'].append(m['iou'])
    star = ''
    if m['miou'] > best_miou:
        best_miou = m['miou']
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        star = ' *'
    if ep % 2 == 0 or ep == EPOCHS - 1:
        print(f'{ep:3d} {tr:11.4f} {m["loss"]:9.4f} {m["pixel_acc"]:8.4f} {m["miou"]:7.4f}   '
              f'{np.round(m["iou"], 3)}{star}')

print(f'\ndone in {time.perf_counter() - t_start:.0f}s | best mIoU {best_miou:.4f}')
model.load_state_dict(best_state)

In [ ]:
final = evaluate_seg(model, val_loader, combined_loss)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].plot(history['train_loss'], marker='.', label='train')
axes[0].plot(history['val_loss'], marker='.', label='val')
axes[0].set_ylabel('CE + Dice'); axes[0].set_title('loss'); axes[0].legend()
axes[1].plot(history['miou'], marker='.', label='mIoU')
axes[1].plot(history['pixel_acc'], marker='.', label='pixel acc')
axes[1].set_ylabel('metric'); axes[1].set_title('mIoU vs pixel accuracy'); axes[1].legend()
iou_hist = np.array(history['iou'])
for c in range(N_CLASSES):
    axes[2].plot(iou_hist[:, c], marker='.', label=CLASS_NAMES[c])
axes[2].set_title('per-class IoU'); axes[2].legend(fontsize=7)
for ax in axes:
    ax.set_xlabel('epoch'); ax.grid(alpha=0.3)
plt.tight_layout()

print(f'final: pixel accuracy {final["pixel_acc"]:.4f} | mIoU {final["miou"]:.4f} | mDice {final["mdice"]:.4f}\n')
print(f'{"class":12} {"IoU":>7} {"Dice":>7} {"pixels":>10}')
for i, name in enumerate(CLASS_NAMES):
    print(f'{name:12} {final["iou"][i]:7.4f} {final["dice"][i]:7.4f} {final["cm"][i].sum():10,d}')
print('\nNotice pixel accuracy started high and barely moved, while mIoU climbed from near 0.')
print('If you had watched only pixel accuracy you would have concluded the model was fine at')
print('epoch 0. This is the whole argument for using the right metric.')

In [ ]:
@torch.no_grad()
def predict_mask(model, img_hwc):
    model.eval()
    x = torch.from_numpy(np.ascontiguousarray(img_hwc.transpose(2, 0, 1)))[None].to(device)
    return model(x).argmax(1)[0].cpu().numpy()

n_show = 5
fig, axes = plt.subplots(4, n_show, figsize=(2.5 * n_show, 10))
for col in range(n_show):
    img, gt = val_ds.items[col + 10]
    pred = predict_mask(model, img)
    err = (pred != gt)
    axes[0, col].imshow(img)
    axes[1, col].imshow(gt, cmap=MASK_CMAP, vmin=0, vmax=3)
    axes[2, col].imshow(pred, cmap=MASK_CMAP, vmin=0, vmax=3)
    axes[3, col].imshow(err, cmap='Reds', vmin=0, vmax=1)
    axes[0, col].set_title(f'IoU {metrics_from_cm(confusion_matrix(gt, pred))["miou"]:.3f}', fontsize=9)
    axes[3, col].set_xlabel(f'{100 * err.mean():.1f}% wrong', fontsize=8)
for row, lab in enumerate(['image', 'ground truth', 'prediction', 'errors']):
    axes[row, 0].set_ylabel(lab, fontsize=9)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('predictions vs ground truth (errors in red)')
plt.tight_layout()
print('The errors concentrate on BOUNDARIES - the hardest pixels, and where the IoU is won or')
print('lost. On real data that is also where the labels themselves are least reliable.')

## 7. Ablation: what do the skip connections actually buy?

Same architecture, same parameter count, same seed - the skips are simply zeroed out. This is the
experiment that makes U-Net's contribution concrete.

In [ ]:
ABL_EPOCHS = 12

def train_variant(use_skips, epochs=ABL_EPOCHS):
    set_seed(0)
    m = UNet(base=16, use_skips=use_skips).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    sc = make_scaler(device.type == 'cuda')
    mious = []
    for ep in range(epochs):
        train_epoch(m, train_loader, opt, combined_loss, sc, sch)
        mious.append(evaluate_seg(m, val_loader, combined_loss)['miou'])
    return m, mious

print(f'training {ABL_EPOCHS} epochs each, identical parameter counts')
model_skip, miou_skip = train_variant(True)
print(f'  with skips   : final mIoU {miou_skip[-1]:.4f}')
model_noskip, miou_noskip = train_variant(False)
print(f'  without skips: final mIoU {miou_noskip[-1]:.4f}')
print(f'\nskip connections are worth {miou_skip[-1] - miou_noskip[-1]:+.4f} mIoU here')
print(f'(parameters: with {sum(p.numel() for p in model_skip.parameters()):,}, '
      f'without {sum(p.numel() for p in model_noskip.parameters()):,} - identical)')

plt.figure(figsize=(6, 3.6))
plt.plot(miou_skip, marker='o', label=f'with skips ({miou_skip[-1]:.3f})')
plt.plot(miou_noskip, marker='s', label=f'no skips ({miou_noskip[-1]:.3f})')
plt.xlabel('epoch'); plt.ylabel('val mIoU'); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.title('skip connection ablation')

img, gt = val_ds.items[3]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
axes[0].imshow(img); axes[0].set_title('image', fontsize=9)
axes[1].imshow(gt, cmap=MASK_CMAP, vmin=0, vmax=3); axes[1].set_title('ground truth', fontsize=9)
axes[2].imshow(predict_mask(model_skip, img), cmap=MASK_CMAP, vmin=0, vmax=3)
axes[2].set_title('with skips: sharp', fontsize=9)
axes[3].imshow(predict_mask(model_noskip, img), cmap=MASK_CMAP, vmin=0, vmax=3)
axes[3].set_title('no skips: blobby', fontsize=9)
for ax in axes: ax.axis('off')
plt.tight_layout()
print('Look at the boundaries. Without skips the decoder has to reconstruct edge detail from a')
print('16x16 feature map, and it cannot - the shapes come out rounded and misplaced.')
print('"My masks are blurry" almost always means "my skip connections are missing or wrong".')

## 8. Loss ablation: CE alone vs CE + Dice

In [ ]:
def train_loss_variant(loss_fn, name, epochs=10):
    set_seed(0)
    m = UNet(base=16).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    sc = make_scaler(device.type == 'cuda')
    hist = []
    for ep in range(epochs):
        train_epoch(m, train_loader, opt, loss_fn, sc, sch)
        hist.append(evaluate_seg(m, val_loader, loss_fn)['miou'])
    print(f'  {name:16} final mIoU {hist[-1]:.4f}')
    return hist

print('10 epochs each:')
h_ce = train_loss_variant(ce, 'CE only')
h_dice = train_loss_variant(dice_loss, 'Dice only')
h_both = train_loss_variant(combined_loss, 'CE + Dice')

plt.figure(figsize=(6, 3.8))
for h, n in [(h_ce, 'CE only'), (h_dice, 'Dice only'), (h_both, 'CE + Dice')]:
    plt.plot(h, marker='o', label=f'{n} ({h[-1]:.3f})')
plt.xlabel('epoch'); plt.ylabel('val mIoU'); plt.legend(fontsize=8); plt.grid(alpha=0.3)
plt.title('loss function ablation')
print('\nOn this balanced-ish synthetic data the difference is small. On real data with a 95%')
print('background class, Dice earns its place - it is a ratio, so it cannot be dominated by')
print('the majority class the way a per-pixel sum can. CE + Dice is the safe default.')

## 9. A real model on a real photo

`torchvision` ships DeepLabV3 pretrained on 21 Pascal VOC classes. This is what production
segmentation looks like: a pretrained ResNet encoder (chapter 5) plus dilated convolutions
(chapter 3) for a large receptive field without losing resolution.

In [ ]:
RUN_PRETRAINED = True     # set False to skip a ~160 MB download

if RUN_PRETRAINED:
    from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
    try:
        w = DeepLabV3_ResNet50_Weights.DEFAULT
        seg_model = deeplabv3_resnet50(weights=w).to(device).eval()
        preprocess = w.transforms()
        voc_classes = w.meta['categories']
        print(f'loaded DeepLabV3-ResNet50 | {len(voc_classes)} classes')
        print(f'parameters: {sum(p.numel() for p in seg_model.parameters()):,} '
              f'(vs our U-Net: {sum(p.numel() for p in model.parameters()):,})')

        import matplotlib.cbook as cbook
        with cbook.get_sample_data('grace_hopper.jpg') as f:
            photo = Image.open(f).convert('RGB')

        x = preprocess(photo)[None].to(device)
        with torch.no_grad():
            out = seg_model(x)['out']                       # (1, 21, H, W) - a dict!
        pred = out.argmax(1)[0].cpu().numpy()

        present = [(voc_classes[i], int((pred == i).sum())) for i in np.unique(pred)]
        print('\nclasses found:', present)

        fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
        axes[0].imshow(photo); axes[0].set_title('input photo')
        axes[1].imshow(pred, cmap='tab20', vmin=0, vmax=20); axes[1].set_title('predicted mask (21 VOC classes)')
        axes[2].imshow(photo.resize((pred.shape[1], pred.shape[0])))
        axes[2].imshow(pred > 0, alpha=0.45, cmap='autumn'); axes[2].set_title('foreground overlay')
        for ax in axes: ax.axis('off')
        plt.tight_layout()
        print('\nNote seg_model(x) returns a DICT with keys "out" and "aux" - a torchvision')
        print('segmentation convention that surprises everyone once. The aux head is a training')
        print('aid (deep supervision) and is ignored at inference.')
    except Exception as e:
        print('skipping pretrained demo:', type(e).__name__, e)
else:
    print('RUN_PRETRAINED = False, skipped')

## What to remember

| Idea | The one-liner |
|---|---|
| The task | per-pixel classification; same loss, one more rank |
| Shapes | logits `(N,C,H,W)` float32, target `(N,H,W)` **int64**, not one-hot |
| Prediction | `logits.argmax(dim=1)` -> `(N,H,W)` |
| Architecture | encoder-decoder + **skip connections** = U-Net |
| Skips | carry high-resolution detail to the decoder; without them, blobby masks |
| Upsampling | `Upsample(bilinear) + Conv` beats `ConvTranspose2d` (no checkerboarding) |
| Mask resizing | **nearest neighbour only** - bilinear invents classes |
| Augmentation | image and mask must get the **identical** transform |
| Metric | per-class IoU and mIoU. **Never** pixel accuracy |
| Metric plumbing | accumulate one confusion matrix; don't average per-batch IoU |
| Loss | `CrossEntropyLoss` + Dice; `ignore_index=255` for void pixels |
| Size constraint | pad input to a multiple of the downsampling factor, predict, crop back |
| Production | pretrained encoder + U-Net decoder (`segmentation_models_pytorch`) |

Now do [`exercises/ex06_segmentation.ipynb`](../exercises/ex06_segmentation.ipynb).

Then read the end of [`docs/06_segmentation.md`](../docs/06_segmentation.md) for where to go next -
and note that the U-Net you just built is, almost unchanged, the denoising network at the heart of
modern diffusion models. This was not a detour.